# Chapter 40: Testing, Middleware & Background Tasks — Colab Notebook

This notebook actually runs the FastAPI/pytest code that the lesson page (`ch40-testing-middleware-background-tasks.html`) shows as reference-only. FastAPI needs a real ASGI runtime and pytest needs a real test-collection process, neither of which Pyodide (the in-browser Python powering the rest of the course) can provide.

This notebook writes real files (`main.py`, `test_main.py`) with `%%writefile` and runs the real `pytest` CLI against them with `!python -m pytest` — the pytest output you see below is genuine, not typed by hand.

Run the cells top to bottom. Install dependencies first if needed:

```
!pip install -q fastapi uvicorn httpx pytest
```


## Setup — the Candidate Scoring API, with middleware, CORS, and a background task

In [1]:
%%writefile main.py
import time
from fastapi import FastAPI, Request, BackgroundTasks
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title="Candidate Scoring API")

CANDIDATES = {
    1: {"name": "Alice", "score": 0.92},
    2: {"name": "Bob",   "score": 0.75},
    3: {"name": "Carol", "score": 0.81},
    4: {"name": "Dave",  "score": 0.60},
    5: {"name": "Eve",   "score": 0.55},
}

SENT_EMAILS = []


@app.middleware("http")
async def add_process_time_header(request: Request, call_next):
    start = time.time()
    response = await call_next(request)
    response.headers["X-Process-Time"] = str(time.time() - start)
    return response


app.add_middleware(
    CORSMiddleware,
    allow_origins=["https://example.com"],
    allow_methods=["GET", "POST"],
)


@app.get("/candidates/{candidate_id}")
def get_candidate(candidate_id: int):
    return CANDIDATES[candidate_id]


@app.get("/candidates")
def list_candidates(min_score: float = 0.0):
    return [c for c in CANDIDATES.values() if c["score"] >= min_score]


def send_welcome_email(name: str):
    SENT_EMAILS.append(name)


@app.post("/signup")
def signup(name: str, background_tasks: BackgroundTasks):
    background_tasks.add_task(send_welcome_email, name)
    return {"status": "signed up"}


Overwriting main.py


In [2]:
from main import app
from fastapi.testclient import TestClient

client = TestClient(app)
print("App loaded with", len(__import__("main").CANDIDATES), "candidates.")


App loaded with 5 candidates.


## 40.1 — pytest + TestClient Fixtures

In [3]:
%%writefile test_main.py
import pytest
from fastapi.testclient import TestClient
from main import app


@pytest.fixture
def client():
    return TestClient(app)


def test_get_candidate_by_id(client):
    response = client.get("/candidates/1")
    assert response.status_code == 200
    assert response.json()["name"] == "Alice"


Overwriting test_main.py


In [4]:
!python -m pytest test_main.py -v --no-header --color=no -p no:warnings -p no:cacheprovider

============================= test session starts ==============================
collecting ... 

collected 1 item                                                               

test_main.py::test_get_candidate_by_id 

PASSED                            [100%]

============================== 1 passed in 0.19s ===============================


## 40.2 — Parametrized Tests

In [5]:
%%writefile -a test_main.py


@pytest.mark.parametrize("candidate_id,expected_name", [
    (1, "Alice"),
    (2, "Bob"),
    (3, "Carol"),
])
def test_get_candidate_name(client, candidate_id, expected_name):
    response = client.get(f"/candidates/{candidate_id}")
    assert response.json()["name"] == expected_name


Appending to test_main.py


In [6]:
!python -m pytest test_main.py -v --no-header --color=no -p no:warnings -p no:cacheprovider

============================= test session starts ==============================
collecting ... 

collected 4 items                                                              

test_main.py::test_get_candidate_by_id PASSED                            [ 25%]
test_main.py::test_get_candidate_name[1-Alice] PASSED                    [ 50%]
test_main.py::test_get_candidate_name[2-Bob] PASSED                      [ 75%]
test_main.py::test_get_candidate_name[3-Carol] PASSED                    [100%]

============================== 4 passed in 0.18s ===============================


## 40.3 — Custom Middleware

In [7]:
response = client.get("/candidates/1")
print("status:", response.status_code)
print("X-Process-Time header present:", "x-process-time" in response.headers)
print("X-Process-Time value:", response.headers.get("x-process-time"))


status: 200
X-Process-Time header present: True
X-Process-Time value: 0.0009272098541259766


## 40.4 — CORS

In [8]:
# A request WITH an Origin header FastAPI's CORSMiddleware allows:
response = client.get("/candidates", headers={"origin": "https://example.com"})
print("status:", response.status_code)
print("Access-Control-Allow-Origin:", response.headers.get("access-control-allow-origin"))

# A request from an origin that was never allow-listed:
response2 = client.get("/candidates", headers={"origin": "https://not-allowed.com"})
print("Access-Control-Allow-Origin (disallowed origin):", response2.headers.get("access-control-allow-origin"))


status: 200
Access-Control-Allow-Origin: https://example.com
Access-Control-Allow-Origin (disallowed origin): None


## 40.5 — Background Tasks

In [9]:
import main as main_module

response = client.post("/signup", params={"name": "Frank"})
print("status:", response.status_code)
print("body:", response.json())
print("SENT_EMAILS after the call returned:", main_module.SENT_EMAILS)


status: 200
body: {'status': 'signed up'}
SENT_EMAILS after the call returned: ['Frank']


## 40.6 — Testing a Background Task's Side Effect

In [10]:
%%writefile -a test_main.py


def test_signup_sends_welcome_email(client):
    response = client.post("/signup", params={"name": "Grace"})
    assert response.status_code == 200
    import main
    assert "Grace" in main.SENT_EMAILS


Appending to test_main.py


In [11]:
!python -m pytest test_main.py -v --no-header --color=no -p no:warnings -p no:cacheprovider

============================= test session starts ==============================
collecting ... 

collected 5 items                                                              

test_main.py::test_get_candidate_by_id PASSED                            [ 20%]
test_main.py::test_get_candidate_name[1-Alice] PASSED                    [ 40%]
test_main.py::test_get_candidate_name[2-Bob] PASSED                      [ 60%]
test_main.py::test_get_candidate_name[3-Carol] PASSED                    [ 80%]
test_main.py::test_signup_sends_welcome_email 

PASSED                     [100%]

============================== 5 passed in 0.18s ===============================


## Mini Project: Testing & Hardening the Candidate Scoring API

Everything from this chapter, run together against the same `main.py`: fixtures, parametrized tests, middleware, CORS, and a background-task test — the full suite passing is the project checklist.

In [12]:
result = get_ipython().getoutput("python -m pytest test_main.py -v --no-header --color=no -p no:warnings -p no:cacheprovider")
print("\n".join(result))

passed_line = [l for l in result if "passed" in l.lower()]
assert passed_line, "Expected a pytest summary line reporting passed tests"
assert "failed" not in passed_line[-1].lower(), "Expected zero failures in the full suite"
print("\nProject checklist: PASSED —", passed_line[-1].strip())


============================= test session starts ==============================
collecting ... collected 5 items

test_main.py::test_get_candidate_by_id PASSED                            [ 20%]
test_main.py::test_get_candidate_name[1-Alice] PASSED                    [ 40%]
test_main.py::test_get_candidate_name[2-Bob] PASSED                      [ 60%]
test_main.py::test_get_candidate_name[3-Carol] PASSED                    [ 80%]
test_main.py::test_signup_sends_welcome_email PASSED                     [100%]

============================== 5 passed in 0.18s ===============================

Project checklist: PASSED — ============================== 5 passed in 0.18s ===============================


### Next: Chapter 41 — FastAPI Capstone: Build & Deploy (Colab, Capstone)